# AETHER — Stage 4 Voice Head (minimal structural probe)

Один codebook Mimi, from-scratch и **необученный** Voice Head поверх Speaker-текста. Это не тест
качества голоса — эмоции, просодия и естественность звучания сознательно вне рамок этого
эксперимента (см. `aether/model/voice_head.py` и `aether/experiments/colab_stage4.py`).

Проверяется ровно одна архитектурная гипотеза: переносится ли существующий commit horizon
(`ChunkState`) на аудио-модальность без изменений — то есть аудио-чанк подчиняется тем же
safety-инвариантам (не коммитится раньше подтверждённого факта, буферизованный синтез корректно
отменяется при replan), что и текстовый чанк.

**GPU:** этот эксперимент не требователен к GPU сверх уже проверенного Qwen3-1.7B — Mimi codec
компактен (десятки миллионов параметров), а сам Voice Head — крошечный (2 слоя, d_model=128) и по
умолчанию работает на CPU. Свободного **T4** в Colab достаточно (как в Stage 1-3); A100 не
обязателен, разве что для более быстрой итерации.

In [ ]:
REPO_URL = "https://github.com/YOUR_USERNAME/YOUR_REPO.git"  # @param {type:"string"}
BRANCH = "main"  # @param {type:"string"}
MODEL_ID = "Qwen/Qwen3-1.7B"  # @param {type:"string"}
TOOL_LATENCIES_MS = "3000,1500,750,300"  # @param {type:"string"}

if "YOUR_USERNAME" in REPO_URL:
    raise ValueError("Укажи настоящий REPO_URL")


In [ ]:
import os, subprocess, sys
from pathlib import Path

subprocess.run(["nvidia-smi"], check=False)
repo_dir = Path("/content/aether")
if (repo_dir / ".git").exists():
    subprocess.run(["git", "-C", str(repo_dir), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(repo_dir)], check=True)
os.chdir(repo_dir)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{repo_dir}[dev,ml,audio]"], check=True)
# `moshi` (the `audio` extra) can pull a newer torch without touching Colab's
# preinstalled torchvision, breaking its ABI (`operator torchvision::nms does
# not exist`) which then makes transformers' AutoModelForCausalLM fail to
# import Qwen3ForCausalLM entirely -- an unrelated vision-utils import path,
# not anything about our text/audio pipeline. Reconcile the trio explicitly.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade", "torch", "torchvision", "torchaudio"],
    check=True,
)
print("Commit:")
subprocess.run(["git", "rev-parse", "HEAD"], check=True)


In [ ]:
artifacts = repo_dir / "artifacts" / "colab-stage4"
artifacts.mkdir(parents=True, exist_ok=True)
tests = subprocess.run(
    [sys.executable, "-m", "pytest", "-q"],
    text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)
(artifacts / "tests.log").write_text(tests.stdout, encoding="utf-8")
print(tests.stdout)
if tests.returncode != 0:
    raise RuntimeError("Tests failed")


In [ ]:
env = os.environ.copy()
env["PYTHONPATH"] = str(repo_dir / "src")
command = [
    sys.executable, "-m", "aether.experiments.colab_stage4",
    "--allow-download",
    "--model", MODEL_ID,
    "--tool-latency-ms", TOOL_LATENCIES_MS,
    "--output-dir", str(artifacts),
]
run = subprocess.run(command, env=env, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
(artifacts / "model_run.log").write_text(run.stdout, encoding="utf-8")
print(run.stdout)
print("Exit code:", run.returncode)


In [ ]:
import json

report_path = artifacts / "report.json"
if report_path.exists():
    report = json.loads(report_path.read_text(encoding="utf-8"))
    print("Status:", report.get("status"))
    print("Scope note:", report.get("scope_note"))
    print("Summary:", json.dumps(report.get("summary", {}), indent=2))
    for run in report.get("runs", []):
        print(
            run.get("scenario"),
            run.get("tool_latency_ms"),
            "PASSED" if run.get("passed") else "FAILED",
            run.get("checks"),
            "wav_files:", list(run.get("wav_files", {}).values()),
        )


## Прослушать сгенерированный аудио-файл (опционально)

Файлы в `wav_files` — сырой выход через реальный (frozen) Mimi decoder, но из **необученного**
Voice Head. Ожидаемо звучит как шум/артефакты, не как речь — целевая проверка здесь
структурная (см. `checks` выше), не перцептивная.

In [ ]:
from IPython.display import Audio, display

if report.get("runs"):
    first_run = report["runs"][0]
    wav_files = list(first_run.get("wav_files", {}).values())
    if wav_files:
        display(Audio(filename=wav_files[0]))


In [ ]:
import shutil
from google.colab import files

archive = shutil.make_archive("/content/aether-colab-stage4-logs", "zip", root_dir=artifacts)
print(archive)
files.download(archive)
